<a href="https://colab.research.google.com/github/wvb20/cv-face-alignment/blob/main/notebooks/02_baseline_sift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === Cell 1: Bootstrap — Drive, repo, src/ imports ===
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

REPO_PATH = '/content/cv-face-alignment'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/wvb20/cv-face-alignment.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from src import config, data, evaluate, features, models, visualise
print('✓ Setup complete')

In [ ]:
# === Cell 2: Load data and apply train/val split ===
images, points = data.load_train()
train_idx, val_idx = data.get_split()

X_train, y_train = images[train_idx], points[train_idx]
X_val,   y_val   = images[val_idx],   points[val_idx]

print(f'Train: {X_train.shape[0]} images')
print(f'Val:   {X_val.shape[0]} images')
print(f'Image shape: {X_train.shape[1:]}, Points shape: {y_train.shape[1:]}')

In [ ]:
# === Cell 3: Mean-shape baseline (the floor everything must beat) ===
ms = features.mean_shape(y_train)
print(f'Mean shape:\n{ms}')

# "Predict" the mean shape for every val image
y_pred_mean = np.tile(ms, (len(X_val), 1, 1))

nme_mean = evaluate.nme(y_pred_mean, y_val)
print(f'\nMean-shape baseline NME: '
      f'mean={nme_mean.mean()*100:.2f}%, '
      f'median={np.median(nme_mean)*100:.2f}%')
print(f'Failure rate (>10%): {evaluate.failure_rate(nme_mean)*100:.1f}%')

In [ ]:
# === Cell 4: Visualise the mean-shape baseline on 6 val examples ===
rng = np.random.default_rng(seed=0)
sample = rng.choice(len(X_val), size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, i in zip(axes.flat, sample):
    ax.imshow(X_val[i])
    ax.scatter(y_val[i, :, 0], y_val[i, :, 1],
               s=80, c='lime', edgecolors='black',
               linewidths=1.5, label='Ground truth', zorder=3)
    ax.scatter(y_pred_mean[i, :, 0], y_pred_mean[i, :, 1],
               s=80, marker='x', c='red',
               linewidths=2.5, label='Mean-shape pred', zorder=4)
    ax.set_title(f'Val #{i}, NME={nme_mean[i]*100:.1f}%')
    ax.axis('off')
axes[0, 0].legend(loc='upper right', fontsize=9)
fig.suptitle('Mean-shape baseline — predictions identical for every face',
             fontsize=13)
fig.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/04_meanshape_baseline.png',
            dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Cell 5: Sanity-check the SIFT feature extractor ===
desc = features.compute_sift_at_points(X_val[0], ms)
print(f'SIFT descriptor for one image:')
print(f'  shape: {desc.shape}  (expect (640,) = 5 landmarks × 128 dims)')
print(f'  dtype: {desc.dtype}')
print(f'  range: [{desc.min():.1f}, {desc.max():.1f}]')

In [ ]:
# === Cell 6: Single-stage Ridge regression on SIFT-at-mean-shape ===
from sklearn.linear_model import Ridge
from tqdm.notebook import tqdm

def extract_features_batch(images, points_per_image):
    """Extract SIFT features at given points for a batch of images."""
    feats = np.zeros((len(images), config.N_LANDMARKS * 128), dtype=np.float32)
    for i, img in enumerate(tqdm(images, desc='SIFT')):
        feats[i] = features.compute_sift_at_points(img, points_per_image)
    return feats

# Extract SIFT-at-mean-shape features for every train and val image
X_train_feats = extract_features_batch(X_train, ms)
X_val_feats   = extract_features_batch(X_val,   ms)

# Target: the offset from mean-shape to ground truth (residual learning)
y_train_offset = (y_train - ms).reshape(len(y_train), -1)   # (N, 10)
y_val_offset   = (y_val   - ms).reshape(len(y_val),   -1)

# Fit Ridge regression
reg = Ridge(alpha=10.0)
reg.fit(X_train_feats, y_train_offset)

# Predict offsets, add mean-shape to get absolute landmark positions
pred_offset = reg.predict(X_val_feats)
y_pred_ridge = pred_offset.reshape(-1, config.N_LANDMARKS, 2) + ms

# Score
nme_ridge = evaluate.nme(y_pred_ridge, y_val)
print(f'\nSingle-stage Ridge on SIFT@mean-shape:')
print(f'  NME mean   = {nme_ridge.mean()*100:.2f}%')
print(f'  NME median = {np.median(nme_ridge)*100:.2f}%')
print(f'  Failure rate (>10%) = {evaluate.failure_rate(nme_ridge)*100:.1f}%')

improvement = (nme_mean.mean() - nme_ridge.mean()) * 100
print(f'\nMean-shape baseline was {nme_mean.mean()*100:.2f}% — '
      f'improvement: {improvement:.2f} pp')

In [ ]:
# === Cell 7: Visualise single-stage Ridge predictions on 6 val examples ===
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, i in zip(axes.flat, sample):  # reuse the sample from Cell 4
    ax.imshow(X_val[i])
    ax.scatter(y_val[i, :, 0], y_val[i, :, 1],
               s=80, c='lime', edgecolors='black',
               linewidths=1.5, label='Ground truth', zorder=3)
    ax.scatter(y_pred_ridge[i, :, 0], y_pred_ridge[i, :, 1],
               s=80, marker='x', c='red',
               linewidths=2.5, label='Ridge pred', zorder=4)
    ax.set_title(f'Val #{i}, NME={nme_ridge[i]*100:.1f}%')
    ax.axis('off')
axes[0, 0].legend(loc='upper right', fontsize=9)
fig.suptitle('Single-stage Ridge on SIFT-at-mean-shape', fontsize=13)
fig.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/05_ridge_baseline.png',
            dpi=120, bbox_inches='tight')
plt.show()